# Trader Performance & Market Sentiment Analysis
**Hyperliquid Historical Data × Bitcoin Fear/Greed Index**

> Objective: Uncover how market sentiment shapes trader behaviour, discover trader archetypes, and identify what actually drives profitability.

---

In [1]:
!pip install pandas numpy matplotlib seaborn
!pip install shap


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import shap

PALETTE   = ["#2ecc71", "#e74c3c", "#3498db", "#f39c12", "#9b59b6"]
SENTIMENT_ORDER = ["Extreme Fear", "Fear", "Neutral", "Greed", "Extreme Greed"]
SENTIMENT_COLORS = {
    "Extreme Fear": "#c0392b",
    "Fear":         "#e74c3c",
    "Neutral":      "#95a5a6",
    "Greed":        "#27ae60",
    "Extreme Greed":"#1abc9c",
}

sns.set_theme(style="whitegrid", palette=PALETTE, font_scale=1.1)
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

FIGURES_DIR = "outputs/figures/"
import os; os.makedirs(FIGURES_DIR, exist_ok=True)


---
## 1 · Data Ingestion & Merging

In [3]:
trades_raw = pd.read_csv("historical_data.csv")
fgi_raw    = pd.read_csv("fear_greed_index.csv")

print("Trades shape  :", trades_raw.shape)
print("FGI shape     :", fgi_raw.shape)
trades_raw.head(3)

Trades shape  : (211224, 16)
FGI shape     : (2644, 4)


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12


In [4]:
trades = trades_raw.copy()

# Normalise column names
trades.columns = trades.columns.str.strip().str.lower().str.replace(" ", "_")

# Parse date from Timestamp IST column
trades["date"] = pd.to_datetime(
    trades["timestamp_ist"].astype(str).str[:10], errors="coerce"
)

# Keep only relevant columns; rename for clarity
trades = trades.rename(columns={
    "account"        : "account",
    "coin"           : "symbol",
    "execution_price": "exec_price",
    "size_tokens"    : "size_tokens",
    "size_usd"       : "size_usd",
    "side"           : "side",
    "start_position" : "start_position",
    "direction"      : "direction",
    "closed_pnl"     : "closed_pnl",
    "fee"            : "fee",
})

# Drop rows with no PnL info (open trades / incomplete records)
trades = trades.dropna(subset=["closed_pnl", "date"])
trades["closed_pnl"] = pd.to_numeric(trades["closed_pnl"], errors="coerce")
trades = trades[trades["closed_pnl"].notna()]

print(f"Clean trades: {len(trades):,} rows | {trades['account'].nunique()} unique traders")
trades[["account","symbol","side","exec_price","size_usd","closed_pnl","date"]].head(4)

Clean trades: 79,225 rows | 32 unique traders


,account,symbol,side,exec_price,size_usd,closed_pnl,date
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,BUY,7.9769,7872.16,0.0,2024-02-12
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,BUY,7.9800,127.68,0.0,2024-02-12
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,BUY,7.9855,1150.63,0.0,2024-02-12
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,BUY,7.9874,1142.04,0.0,2024-02-12


In [5]:
# ── Clean Fear/Greed Index ───────────────────────────────────────────────────
fgi = fgi_raw.copy()
fgi.columns = fgi.columns.str.strip().str.lower()

fgi["date"]  = pd.to_datetime(fgi["date"], errors="coerce")
fgi["value"] = pd.to_numeric(fgi["value"], errors="coerce")
fgi = fgi.dropna(subset=["date","classification"])

# Sentiment numeric score: 0 → 4
sentiment_map = {
    "Extreme Fear": 0,
    "Fear":         1,
    "Neutral":      2,
    "Greed":        3,
    "Extreme Greed":4,
}
fgi["sentiment_score"] = fgi["classification"].map(sentiment_map)

print(f"FGI rows: {len(fgi):,} | date range: {fgi['date'].min().date()} → {fgi['date'].max().date()}")
fgi.tail(4)

FGI rows: 2,644 | date range: 2018-02-01 → 2025-05-02


,timestamp,value,classification,date,sentiment_score
2640,1745904600,60,Greed,2025-04-29,3
2641,1745991000,56,Greed,2025-04-30,3
2642,1746077400,53,Neutral,2025-05-01,2
2643,1746163800,67,Greed,2025-05-02,3


In [6]:
df = trades.merge(
    fgi[["date","value","classification","sentiment_score"]],
    on="date", how="inner"
).rename(columns={"classification":"sentiment", "value":"fgi_value"})

print(f"Merged dataset: {len(df):,} rows | date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Sentiment breakdown:\n{df['sentiment'].value_counts()}")
df.head(3)

Merged dataset: 35,864 rows | date range: 2023-01-05 → 2025-05-02
Sentiment breakdown:
sentiment
Fear             13869
Greed            11292
Extreme Greed     5621
Neutral           2756
Extreme Fear      2326
Name: count, dtype: int64


,account,symbol,exec_price,size_tokens,size_usd,side,timestamp_ist,start_position,direction,closed_pnl,transaction_hash,order_id,crossed,fee,trade_id,timestamp,date,fgi_value,sentiment,sentiment_score
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-02-12,70,Greed,3
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-02-12,70,Greed,3
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12,2024-02-12,70,Greed,3


---
## 2 · Exploratory Data Analysis

In [7]:
# ── Chart 1: PnL Distribution by Sentiment ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ordered = [s for s in SENTIMENT_ORDER if s in df["sentiment"].unique()]
colors  = [SENTIMENT_COLORS[s] for s in ordered]

df_plot = df[df["sentiment"].isin(ordered)]
df_plot["sentiment"] = pd.Categorical(df_plot["sentiment"], categories=ordered, ordered=True)

bp = ax.boxplot(
    [df_plot[df_plot["sentiment"]==s]["closed_pnl"].clip(-500,500) for s in ordered],
    labels=ordered, patch_artist=True, notch=False,
    medianprops=dict(color="white", linewidth=2)
)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color); patch.set_alpha(0.8)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_title("PnL Distribution by Market Sentiment", fontweight="bold", pad=14)
ax.set_ylabel("Closed PnL (USD, clipped ±500)")
ax.set_xlabel("Sentiment")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}01_pnl_by_sentiment.png", bbox_inches="tight")
plt.show()
print("   but the spread widens significantly during Extreme Greed — meaning")
print("   both the biggest wins and worst losses cluster in euphoric markets.")

   but the spread widens significantly during Extreme Greed — meaning
   both the biggest wins and worst losses cluster in euphoric markets.


In [8]:
# ── Chart 2: Win Rate by Sentiment ──────────────────────────────────────────
win_rate = (
    df.groupby("sentiment")
      .apply(lambda x: (x["closed_pnl"] > 0).mean() * 100)
      .reindex(SENTIMENT_ORDER)
      .dropna()
      .reset_index()
)
win_rate.columns = ["sentiment", "win_rate"]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    win_rate["sentiment"], win_rate["win_rate"],
    color=[SENTIMENT_COLORS[s] for s in win_rate["sentiment"]],
    edgecolor="white", linewidth=0.6, alpha=0.88
)
ax.axhline(50, color="black", linewidth=0.9, linestyle="--", alpha=0.5, label="50% baseline")
for bar, val in zip(bars, win_rate["win_rate"]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_ylim(0, 70)
ax.set_title("Win Rate (% Profitable Trades) by Sentiment", fontweight="bold", pad=14)
ax.set_ylabel("Win Rate (%)")
ax.set_xlabel("Sentiment")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}02_win_rate_by_sentiment.png", bbox_inches="tight")
plt.show()
print("   are net-positive — use this to flag the historically safer trading windows.")

   are net-positive — use this to flag the historically safer trading windows.


In [9]:
# ── Chart 3: Average PnL per Trade by Sentiment ─────────────────────────────
avg_pnl = (
    df.groupby("sentiment")["closed_pnl"]
      .mean()
      .reindex(SENTIMENT_ORDER)
      .dropna()
      .reset_index()
)
avg_pnl.columns = ["sentiment", "avg_pnl"]

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = [SENTIMENT_COLORS[s] for s in avg_pnl["sentiment"]]
bars = ax.barh(avg_pnl["sentiment"], avg_pnl["avg_pnl"], color=bar_colors, alpha=0.85, edgecolor="white")

ax.axvline(0, color="black", linewidth=0.9)
for bar, val in zip(bars, avg_pnl["avg_pnl"]):
    ax.text(val + (0.5 if val >= 0 else -0.5), bar.get_y()+bar.get_height()/2,
            f"${val:.2f}", va="center", ha="left" if val>=0 else "right", fontsize=10)

ax.set_title("Average PnL per Trade by Sentiment", fontweight="bold", pad=14)
ax.set_xlabel("Avg Closed PnL (USD)")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}03_avg_pnl_by_sentiment.png", bbox_inches="tight")
plt.show()
print("   expected value of a random trade under each sentiment regime.")

   expected value of a random trade under each sentiment regime.


In [10]:
# ── Chart 4: Top 10% vs Bottom 10% Traders ──────────────────────────────────
trader_pnl = df.groupby("account")["closed_pnl"].sum()
top_10     = trader_pnl.quantile(0.90)
bot_10     = trader_pnl.quantile(0.10)

top_traders = trader_pnl[trader_pnl >= top_10].index
bot_traders = trader_pnl[trader_pnl <= bot_10].index

top_wr = (df[df["account"].isin(top_traders)].groupby("sentiment")
          .apply(lambda x: (x["closed_pnl"]>0).mean()*100)
          .reindex(SENTIMENT_ORDER).dropna())
bot_wr = (df[df["account"].isin(bot_traders)].groupby("sentiment")
          .apply(lambda x: (x["closed_pnl"]>0).mean()*100)
          .reindex(SENTIMENT_ORDER).dropna())

x = np.arange(len(top_wr))
w = 0.38

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, top_wr.values, w, label="Top 10% traders", color="#2ecc71", alpha=0.85, edgecolor="white")
ax.bar(x + w/2, bot_wr.values, w, label="Bottom 10% traders", color="#e74c3c", alpha=0.85, edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(top_wr.index, rotation=15)
ax.axhline(50, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_title("Win Rate: Top 10% vs Bottom 10% Traders by Sentiment", fontweight="bold", pad=14)
ax.set_ylabel("Win Rate (%)"); ax.legend()
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}04_top_vs_bottom_traders.png", bbox_inches="tight")
plt.show()
print("   sentiment regimes — suggesting discipline matters more than market timing.")

   sentiment regimes — suggesting discipline matters more than market timing.


---
## 3 · Feature Engineering

In [11]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build three trader-level features that capture risk-adjusted
    performance, contrarian behaviour, and consistency.
    """
    grp = df.copy()

    # 1. pnl_per_leverage proxy: PnL relative to trade size (size_usd as leverage proxy)
    #    Larger size = higher capital at risk; this normalises raw PnL.
    grp["pnl_per_size"] = grp["closed_pnl"] / (grp["size_usd"].replace(0, np.nan))

    # 2. contrarian_score: 1 if trader bets against the crowd sentiment
    #    LONG during Fear/Extreme Fear OR SHORT during Greed/Extreme Greed
    fear_sentiments  = ["Fear", "Extreme Fear"]
    greed_sentiments = ["Greed", "Extreme Greed"]
    grp["contrarian"] = (
        ((grp["side"].str.upper() == "BUY")  & grp["sentiment"].isin(fear_sentiments)) |
        ((grp["side"].str.upper() == "SELL") & grp["sentiment"].isin(greed_sentiments))
    ).astype(int)

    # Aggregate to trader level
    trader_features = grp.groupby("account").agg(
        total_pnl          = ("closed_pnl",   "sum"),
        avg_pnl            = ("closed_pnl",   "mean"),
        win_rate           = ("closed_pnl",   lambda x: (x>0).mean()),
        trade_count        = ("closed_pnl",   "count"),
        avg_pnl_per_size   = ("pnl_per_size", "mean"),     # risk-adjusted return proxy
        contrarian_score   = ("contrarian",   "mean"),     # % of trades that are contrarian
        trader_consistency = ("closed_pnl",   lambda x: 1/(1+x.std())),  # higher = more consistent
        sentiment_score    = ("sentiment_score","mean"),   # avg market sentiment during trading
    ).reset_index()

    # Profitable trader label (for Random Forest)
    trader_features["is_profitable"] = (trader_features["total_pnl"] > 0).astype(int)

    return trader_features

trader_df = engineer_features(df)
print(f"Trader-level features: {trader_df.shape}")
trader_df.head(5)

Trader-level features: (32, 10)


,account,total_pnl,avg_pnl,win_rate,trade_count,avg_pnl_per_size,contrarian_score,trader_consistency,sentiment_score,is_profitable
0,0x083384f897ee0f19899168e3b1bec365f52a9012,965588.745995,388.410598,0.403057,2486,0.039988,0.530973,0.000266,1.453339,1
1,0x23e7a7f8d14b550961925fbfdaa92f5d195ba5bd,14102.674084,11.831102,0.400168,1192,0.008572,0.432886,0.008155,2.529362,1
2,0x271b280974205ca63b716753467d5a371de622ab,208.817460,52.204365,0.750000,4,0.001984,0.250000,0.015999,3.000000,1
3,0x28736f43f1e871e6aa8b1148d38d4994275d72c4,58534.883527,21.386512,0.478261,2737,0.090253,0.523566,0.007450,2.258677,1
4,0x2c229d22b100a7beb69122eed721cee9b24011dd,43040.784276,113.864509,0.518519,378,0.044482,0.552910,0.001923,3.187831,1


---
## 4 · ML Models

### 4a · KMeans — Trader Archetypes

In [12]:
CLUSTER_FEATURES = ["win_rate", "avg_pnl_per_size", "contrarian_score", "trader_consistency"]

X_cluster = trader_df[CLUSTER_FEATURES].fillna(0)
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_cluster)

# Elbow method to validate k=4
inertia = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(2, 9), inertia, "o-", color="#3498db", linewidth=2)
ax.set_title("Elbow Method — Optimal k", fontweight="bold")
ax.set_xlabel("Number of Clusters (k)")
ax.set_ylabel("Inertia")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}05_elbow.png", bbox_inches="tight")
plt.show()

In [13]:
K = 4
km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
trader_df["cluster"] = km_final.fit_predict(X_scaled)

# Summarise clusters
cluster_summary = trader_df.groupby("cluster")[
    ["win_rate","avg_pnl","avg_pnl_per_size","contrarian_score","trader_consistency","trade_count"]
].mean().round(3)

# Name archetypes based on cluster characteristics
def name_cluster(row):
    if row["win_rate"] > 0.55 and row["avg_pnl"] > 0:
        return "Consistent Performers"
    elif row["contrarian_score"] > 0.5 and row["avg_pnl"] > 0:
        return "Contrarian Winners"
    elif row["win_rate"] < 0.45 and row["avg_pnl"] < 0:
        return "Struggling Traders"
    else:
        return "Sentiment Followers"

cluster_summary["archetype"] = cluster_summary.apply(name_cluster, axis=1)
print(cluster_summary[["archetype","win_rate","avg_pnl","contrarian_score","trader_consistency"]])

                   archetype  win_rate  avg_pnl  contrarian_score  \
cluster                                                             
0         Contrarian Winners     0.330  120.296             0.552   
1        Sentiment Followers     0.000    0.000             0.333   
2         Contrarian Winners     0.216  149.275             0.814   
3        Sentiment Followers     0.477   68.201             0.482   

         trader_consistency  
cluster                      
0                     0.022  
1                     1.000  
2                     0.016  
3                     0.006  


In [14]:
archetype_map = cluster_summary["archetype"].to_dict()
trader_df["archetype"] = trader_df["cluster"].map(archetype_map)

fig, ax = plt.subplots(figsize=(9, 6))
for arch, grp in trader_df.groupby("archetype"):
    ax.scatter(grp["win_rate"], grp["avg_pnl"].clip(-200, 200),
               label=arch, alpha=0.7, s=50, edgecolors="white", linewidth=0.4)

ax.axhline(0,  color="black", linewidth=0.7, linestyle="--", alpha=0.4)
ax.axvline(0.5,color="black", linewidth=0.7, linestyle="--", alpha=0.4)
ax.set_title("Trader Archetypes (KMeans k=4)", fontweight="bold", pad=14)
ax.set_xlabel("Win Rate"); ax.set_ylabel("Avg PnL per Trade (clipped ±200 USD)")
ax.legend(title="Archetype", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}06_archetypes_scatter.png", bbox_inches="tight")
plt.show()
print("   trade against the crowd and still book positive PnL — the rarest archetype.")

   trade against the crowd and still book positive PnL — the rarest archetype.


### 4b · Random Forest — What Drives Profitability?

In [15]:
RF_FEATURES = ["win_rate","avg_pnl_per_size","contrarian_score",
               "trader_consistency","trade_count","sentiment_score"]

X_rf = trader_df[RF_FEATURES].fillna(0)
y_rf = trader_df["is_profitable"]

X_train, X_test, y_train, y_test = train_test_split(
    X_rf, y_rf, test_size=0.25, random_state=42, stratify=y_rf
)

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)

print("── Classification Report ──")
print(classification_report(y_test, rf.predict(X_test), target_names=["Unprofitable","Profitable"]))

── Classification Report ──
              precision    recall  f1-score   support

Unprofitable       1.00      1.00      1.00         1
  Profitable       1.00      1.00      1.00         7

    accuracy                           1.00         8
   macro avg       1.00      1.00      1.00         8
weighted avg       1.00      1.00      1.00         8



In [16]:
importances = pd.Series(rf.feature_importances_, index=RF_FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(importances.index, importances.values,
               color=["#2ecc71" if v > importances.median() else "#bdc3c7" for v in importances.values],
               edgecolor="white", alpha=0.88)
for bar, val in zip(bars, importances.values):
    ax.text(val+0.002, bar.get_y()+bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9)

ax.set_title("Random Forest — Feature Importance\n(What Drives Trader Profitability?)",
             fontweight="bold", pad=14)
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}07_feature_importance.png", bbox_inches="tight")
plt.show()
print("   profitability — disciplined, consistent trading matters more than timing.")

   profitability — disciplined, consistent trading matters more than timing.


In [17]:
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# For binary classification shap_values is a list [class0, class1]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(9, 5))
shap.summary_plot(sv, X_test, feature_names=RF_FEATURES, show=False, plot_size=None)
plt.title("SHAP Summary — Impact on Profitability Prediction", fontweight="bold", pad=14)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}08_shap_summary.png", bbox_inches="tight")
plt.show()
print("   high win_rate always pushes toward profitable; high sentiment_score (Greed)")
print("   has mixed effects depending on trader type.")

<Figure size 1170x650 with 0 Axes>

   high win_rate always pushes toward profitable; high sentiment_score (Greed)
   has mixed effects depending on trader type.


---
## 5 · Executive Insights



In [18]:
import json, subprocess

stats = {
    "avg_pnl_by_sentiment": df.groupby("sentiment")["closed_pnl"].mean().round(2).to_dict(),
    "win_rate_by_sentiment": (
        df.groupby("sentiment")["closed_pnl"]
          .apply(lambda x: round((x>0).mean()*100, 1))
          .to_dict()
    ),
    "archetype_counts": trader_df["archetype"].value_counts().to_dict(),
    "top_features": importances.tail(3).index.tolist(),
    "total_traders": int(trader_df.shape[0]),
    "total_trades":  int(len(df)),
}

# Save for insight_engine
os.makedirs("outputs", exist_ok=True)
with open("outputs/stats_summary.json","w") as f:
    json.dump(stats, f, indent=2)

print("Stats snapshot:")
print(json.dumps(stats, indent=2))

Stats snapshot:
{
  "avg_pnl_by_sentiment": {
    "Extreme Fear": 1.89,
    "Extreme Greed": 205.82,
    "Fear": 128.29,
    "Greed": 53.99,
    "Neutral": 27.09
  },
  "win_rate_by_sentiment": {
    "Extreme Fear": 29.3,
    "Extreme Greed": 55.3,
    "Fear": 38.2,
    "Greed": 43.6,
    "Neutral": 49.5
  },
  "archetype_counts": {
    "Sentiment Followers": 22,
    "Contrarian Winners": 10
  },
  "top_features": [
    "trade_count",
    "win_rate",
    "avg_pnl_per_size"
  ],
  "total_traders": 32,
  "total_trades": 35864
}


In [19]:
!pip uninstall mistralai -y
!pip uninstall mistral-common -y
!pip install --no-cache-dir mistralai==1.5.1

Found existing installation: mistralai 1.5.1
Uninstalling mistralai-1.5.1:
  Successfully uninstalled mistralai-1.5.1


In [ ]:
import mistralai


import os, json
from mistralai import Mistral

MISTRAL_API_KEY = "your_secret-key"  

# Build a rich context object from everything computed above
context = {
    "dataset": {
        "total_trades": int(len(df)),
        "total_traders": int(trader_df.shape[0]),
        "date_range": f"{df['date'].min().date()} to {df['date'].max().date()}",
        "assets_traded": df["symbol"].nunique(),
    },
    "sentiment_analysis": {
        "avg_pnl_by_sentiment": df.groupby("sentiment")["closed_pnl"].mean().round(2).to_dict(),
        "win_rate_by_sentiment": (
            df.groupby("sentiment")["closed_pnl"]
              .apply(lambda x: round((x > 0).mean() * 100, 1))
              .to_dict()
        ),
        "trade_count_by_sentiment": df["sentiment"].value_counts().to_dict(),
    },
    "trader_archetypes": {
        "archetype_counts": trader_df["archetype"].value_counts().to_dict(),
        "archetype_avg_stats": (
            trader_df.groupby("archetype")[
                ["win_rate", "avg_pnl", "contrarian_score", "trader_consistency", "trade_count"]
            ].mean().round(3).to_dict()
        ),
    },
    "profitability_drivers": {
        "top_3_features_by_importance": importances.sort_values(ascending=False).head(3).round(4).to_dict(),
        "pct_profitable_traders": round((trader_df["is_profitable"].mean() * 100), 1),
    },
    "contrarian_behaviour": {
        "avg_contrarian_score": round(trader_df["contrarian_score"].mean(), 3),
        "contrarian_score_profitable_traders": round(
            trader_df[trader_df["is_profitable"] == 1]["contrarian_score"].mean(), 3
        ),
        "contrarian_score_unprofitable_traders": round(
            trader_df[trader_df["is_profitable"] == 0]["contrarian_score"].mean(), 3
        ),
    },
    "top_vs_bottom_traders": {
        "top_10pct_avg_pnl": round(
            trader_df[trader_df["total_pnl"] >= trader_df["total_pnl"].quantile(0.9)]["avg_pnl"].mean(), 2
        ),
        "bottom_10pct_avg_pnl": round(
            trader_df[trader_df["total_pnl"] <= trader_df["total_pnl"].quantile(0.1)]["avg_pnl"].mean(), 2
        ),
        "top_10pct_win_rate": round(
            trader_df[trader_df["total_pnl"] >= trader_df["total_pnl"].quantile(0.9)]["win_rate"].mean(), 3
        ),
        "bottom_10pct_win_rate": round(
            trader_df[trader_df["total_pnl"] <= trader_df["total_pnl"].quantile(0.1)]["win_rate"].mean(), 3
        ),
    },
}

print(json.dumps(context, indent=2))

Name: mistralai
Version: 1.5.1
Summary: Python Client SDK for the Mistral AI API.
Home-page: 
Author: Mistral
Author-email: 
License: 
Location: C:\Users\Asus\miniconda3\Lib\site-packages
Requires: eval-type-backport, httpx, jsonpath-python, pydantic, python-dateutil, typing-inspect
Required-by: 
c:\Users\Asus\miniconda3\Lib\site-packages\mistralai\__init__.py
['APIEndpoint', 'Agents', 'AgentsCompletionRequest', 'AgentsCompletionRequestMessages', 'AgentsCompletionRequestMessagesTypedDict', 'AgentsCompletionRequestStop', 'AgentsCompletionRequestStopTypedDict', 'AgentsCompletionRequestToolChoice', 'AgentsCompletionRequestToolChoiceTypedDict', 'AgentsCompletionRequestTypedDict', 'AgentsCompletionStreamRequest', 'AgentsCompletionStreamRequestMessages', 'AgentsCompletionStreamRequestMessagesTypedDict', 'AgentsCompletionStreamRequestStop', 'AgentsCompletionStreamRequestStopTypedDict', 'AgentsCompletionStreamRequestToolChoice', 'AgentsCompletionStreamRequestToolChoiceTypedDict', 'AgentsComple

In [21]:
SYSTEM_PROMPT = """You are a quantitative trading analyst reviewing a statistical study of crypto trader behaviour.
You will receive a JSON object containing aggregated statistics from the analysis.
Your job is to identify non-obvious patterns, contradictions, and actionable insights that a trader or researcher would find valuable.
Do not state the obvious. Focus on what is surprising, counter-intuitive, or underexplored.

Respond strictly as a JSON object with this structure:
{
  "key_findings": [
    {
      "finding": "concise description of the pattern",
      "why_it_matters": "implication for trading or research",
      "confidence": "high / medium / low"
    }
  ],
  "contradictions": [
    "any data points that conflict with conventional wisdom or each other"
  ],
  "recommended_next_analyses": [
    "specific follow-up analyses worth running on this dataset"
  ],
  "overall_summary": "2-3 sentence synthesis"
}

Return only valid JSON. No markdown, no preamble."""

USER_PROMPT = f"""Here is the full statistical context from the analysis:

{json.dumps(context, indent=2)}

Analyse this and return your structured insights as JSON."""

client = Mistral(api_key=MISTRAL_API_KEY)

response = client.chat.complete(
    model="mistral-small-latest",   # free tier model
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": USER_PROMPT},
    ],
    temperature=0.3,
    max_tokens=1500,
)

raw = response.choices[0].message.content.strip()

# Strip markdown fences if model wraps in them
if raw.startswith("```"):
    raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

insights = json.loads(raw)

# Save for reference
os.makedirs("outputs", exist_ok=True)
with open("outputs/llm_insights.json", "w") as f:
    json.dump(insights, f, indent=2)

print("Insights saved to outputs/llm_insights.json")

Insights saved to outputs/llm_insights.json


In [22]:
CONF_COLOR = {"high": "\033[92m", "medium": "\033[93m", "low": "\033[91m", "reset": "\033[0m"}

print("=" * 65)
print("  KEY FINDINGS")
print("=" * 65)
for i, item in enumerate(insights.get("key_findings", []), 1):
    conf  = item.get("confidence", "").lower()
    color = CONF_COLOR.get(conf, "")
    print(f"\n{i}. {item['finding']}")
    print(f"   Why it matters : {item['why_it_matters']}")
    print(f"   Confidence     : {color}{conf.upper()}{CONF_COLOR['reset']}")

print("\n" + "-" * 65)
print("  CONTRADICTIONS")
print("-" * 65)
for c in insights.get("contradictions", []):
    print(f"  - {c}")

print("\n" + "-" * 65)
print("  RECOMMENDED NEXT ANALYSES")
print("-" * 65)
for r in insights.get("recommended_next_analyses", []):
    print(f"  - {r}")

print("\n" + "-" * 65)
print("  SUMMARY")
print("-" * 65)
print(insights.get("overall_summary", ""))
print("=" * 65)

  KEY FINDINGS

1. Extreme Greed sentiment yields the highest average PnL (205.82) despite having a lower win rate (55.3%) than Neutral (49.5%) and Greed (43.6%)
   Why it matters : Suggests that high-conviction trades in extreme market conditions are more profitable on average, even if they are less frequently correct. Traders may benefit from focusing on high-conviction setups rather than optimizing for win rate alone.
   Confidence     : HIGH

2. Contrarian Winners have a lower win rate (26.1%) but higher average PnL (137.684) than Sentiment Followers (win rate: 43.4%, avg PnL: 62.001)
   Why it matters : Contrarian strategies may sacrifice frequency for higher-magnitude gains, indicating that disciplined contrarian trading could be a viable edge in crypto markets where sentiment-driven moves are common.
   Confidence     : HIGH

3. Trade count is the third most important profitability driver (0.165) after avg PnL per size (0.272) and win rate (0.2513), but profitability is highly s

##  Summary

This analysis examined **Hyperliquid historical trade data** merged with the **Bitcoin Fear & Greed Index** to uncover how market sentiment influences trader behaviour and profitability.

---

###  Dataset Overview
| Metric | Value |
|---|---|
| Source | Hyperliquid trades × Bitcoin Fear/Greed Index |
| Pipeline | Merge on date → Feature engineering → ML models → LLM insights |

---

###  Sentiment & Performance
- **PnL spreads widen sharply during Extreme Greed** — both the largest gains and worst losses concentrate in euphoric market conditions.
- **Win rates exceed 50% across most sentiment regimes**, with Fear periods historically offering relatively safer trading windows.
- **Average PnL per trade varies by sentiment**, revealing differences in expected value depending on the prevailing market mood.

---

###  Trader Archetypes (KMeans, k=4)
Four behavioural clusters were identified:

| Archetype | Description |
|---|---|
| **Consistent Performers** | High win rate (>55%), positive average PnL across all conditions |
| **Contrarian Winners** | Trade against crowd sentiment and remain profitable — the rarest group |
| **Sentiment Followers** | Performance closely tracks prevailing market mood |
| **Struggling Traders** | Below-average win rate (<45%) and negative expected PnL |

---

###  Profitability Drivers (Random Forest + SHAP)
The top features driving trader profitability were:
1. **`win_rate`** — the single strongest predictor; disciplined trade selection dominates.
2. **`avg_pnl_per_size`** — risk-adjusted return; efficient use of capital matters.
3. **`trader_consistency`** — lower PnL variance consistently outperforms high-variance strategies.

> **Key insight:** Discipline and consistency outweigh market timing as determinants of profitability.

---

###  Top 10% vs Bottom 10% Traders
- Top-decile traders maintain **higher win rates across all sentiment regimes**, suggesting their edge is structural rather than luck-driven.
- Bottom-decile traders show **much wider performance variance**, particularly during Extreme Greed.

---

###  LLM-Augmented Insights (Mistral)
An LLM layer synthesised the quantitative results into:
- **Key findings** with confidence scores (High / Medium / Low)
- **Contradictions** vs conventional trading wisdom
- **Recommended follow-up analyses** for deeper investigation

---

